# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [1]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [2]:
# 1. 创建三个可学习参数 w1,w2,w3，开启梯度追踪
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
w3 = torch.tensor(0.0, requires_grad=True)

lr = 0.02          # 学习率，可以自己微调
epoch_num = 8000   # 训练轮数

for epoch in range(epoch_num):
    # 线索1：预测函数 ŷ = w1*x + w2*x² + w3
    y_pred = w1 * x + w2 * (x ** 2) + w3

    # 线索2：平均绝对误差 MAE，不能用 MSE！
    loss = torch.mean(torch.abs(y - y_pred))

    # 反向传播求梯度
    loss.backward()

    # 梯度下降更新参数（手动 SGD）
    with torch.no_grad():
        w1 -= lr * w1.grad
        w2 -= lr * w2.grad
        w3 -= lr * w3.grad

        # 清空梯度，防止累加
        w1.grad.zero_()
        w2.grad.zero_()
        w3.grad.zero_()

# --------训练结束，输出结果--------
print("=====训练学到的三个参数=====")
print(f"w1 = {w1.item():.4f}")
print(f"w2 = {w2.item():.4f}")
print(f"w3 = {w3.item():.4f}")

# 用最终参数计算所有样本预测值
final_y_pred = w1 * x + w2 * (x ** 2) + w3
final_loss = torch.mean(torch.abs(y - final_y_pred))

print("\n=====每个样本：x, 真实y, 预测ŷ=====")
for xi, yi, ypi in zip(x, y, final_y_pred):
    print(f"x={xi.item():.1f}, y_true={yi.item():.4f}, y_pred={ypi.item():.4f}")

print(f"\n最终平均绝对误差 loss = {final_loss.item():.4f}")

=====训练学到的三个参数=====
w1 = 1.5044
w2 = -0.5111
w3 = 1.9844

=====每个样本：x, 真实y, 预测ŷ=====
x=-2.0, y_true=-3.0000, y_pred=-3.0689
x=-1.5, y_true=-1.3750, y_pred=-1.4222
x=-1.0, y_true=0.0000, y_pred=-0.0311
x=-0.5, y_true=1.1250, y_pred=1.1044
x=0.0, y_true=2.0000, y_pred=1.9844
x=0.5, y_true=2.6250, y_pred=2.6089
x=1.0, y_true=3.0000, y_pred=2.9778
x=1.5, y_true=3.1250, y_pred=3.0911
x=2.0, y_true=3.0000, y_pred=2.9489

最终平均绝对误差 loss = 0.0341


## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。

### 我对作业 01 的训练代码做了哪些修改

1. 参数变多了：作业 01 只有 w 一个参数，本题有 w1、w2、w3 三个待学习参数。
2. 预测函数变了：由 ŷ = x·w² 改为 ŷ = w1·x + w2·x² + w3。
3. 误差的求法有变化：作业 01 是单个样本的误差，本题对所有样本取平均，使用平均绝对误差（MAE），没有换成平方误差。

### 如何判断这次训练有效

训练结束后最终平均绝对误差很小（约 0.034），每个样本的预测值 ŷ 都贴近真实值 y，说明三个参数已经拟合到给定数据，训练是有效的。